pydantic library

In [9]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(ge=0)  # ge=0 means greater than or equal to 0
    email: str

# 1. Parsing and validating raw JSON/dict data:
raw_input = {"name": "Alice", "age": "25", "email": "alice@example.com"}
user = User(**raw_input)

print(user.age)        # 25 (automatically converted from string to int)
print(type(user.age))  # <class 'int'>

# 2. Exporting back to dict or JSON:
user_dict = user.model_dump()       # {'name': 'Alice', 'age': 25, 'email': 'alice@example.com'}
user_json = user.model_dump_json()  # '{"name":"Alice","age":25,"email":"alice@example.com"}'

25
<class 'int'>


In [ ]:
"""OpenAI Structured Outputs backend for paper evidence extraction."""

from __future__ import annotations

import os
import logging
import re
from time import perf_counter
from concurrent.futures import Future, ThreadPoolExecutor
from pathlib import Path
from threading import RLock
from typing import Any, Literal, Sequence

from pydantic import BaseModel, ConfigDict, Field, ValidationError

from src.config import cache_dir, openai_api_key, openai_extraction_model
from src.models.paper import Paper

from .evidence import EvidenceItem, LimitationEvidence, PaperEvidence, StudyType, canonical_evidence_key
from .store import EvidenceStore


LOGGER = logging.getLogger(__name__)


# Increment when the structured extraction contract or its compatibility
# assumptions change. Old cache rows remain harmless misses after a bump.
EVIDENCE_SCHEMA_VERSION = 4


class PaperExtractionError(RuntimeError):
    """Raised when a paper cannot be converted into structured evidence."""


class _LimitationClaim(EvidenceItem):
    author_stated: bool


class _MethodClaim(EvidenceItem):
    role: Literal["primary", "supporting", "comparison"]


class _ExtractionResult(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    research_objective: EvidenceItem | None = Field(
        default=None,
        description="The main research problem, objective, or question explicitly investigated by the paper.",
    )
    population_or_setting: list[EvidenceItem] = Field(
        default_factory=list,
        description="Explicit populations, application domains, environments, or experimental settings.",
    )
    method_or_intervention: list[_MethodClaim] = Field(
        default_factory=list,
        description=(
            "Methods, models, algorithms, interventions, or architectures actually studied. "
            "Each claim must be marked primary, supporting, or comparison."
        ),
    )
    comparison_or_baseline: list[EvidenceItem] = Field(
        default_factory=list,
        description="Methods or systems explicitly compared with or evaluated against the focal method.",
    )
    data_or_modality: list[EvidenceItem] = Field(
        default_factory=list,
        description=(
            "Input data, measurements, signals, sensing modalities, source data, "
            "or input representations explicitly used by the study."
        ),
    )
    datasets: list[EvidenceItem] = Field(
        default_factory=list,
        description="Named datasets or sufficiently specific dataset descriptions.",
    )
    sample_size: EvidenceItem | None = Field(
        default=None,
        description="An explicit numerical sample count such as images, participants, documents, records, or examples.",
    )
    evaluation_metrics: list[EvidenceItem] = Field(
        default_factory=list,
        description="Explicit evaluation metrics such as accuracy, F1, AUC, BLEU, ROUGE, or NDCG.",
    )
    main_findings: list[EvidenceItem] = Field(
        default_factory=list,
        description="Major empirical findings or conclusions explicitly reported by the authors.",
    )
    constraints: list[EvidenceItem] = Field(
        default_factory=list,
        description=(
            "Explicit experimental, data, resource, deployment, generalization, or evaluation constraints. "
            "The value must describe the constraint itself, never merely name a method or model."
        ),
    )
    limitations: list[_LimitationClaim] = Field(
        default_factory=list,
        description="Limitations or weaknesses explicitly attributed to the paper by its authors.",
    )
    future_work: list[EvidenceItem] = Field(
        default_factory=list,
        description="Concrete future research directions explicitly proposed by the authors.",
    )
    study_type: StudyType = Field(
        default="other",
        description="Study classification based only on the supplied title and abstract.",
    )
    extraction_confidence: float = Field(ge=0.0, le=1.0)


class _BatchPaperExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    paper_id: str = Field(min_length=1)
    evidence: _ExtractionResult


class _BatchExtractionResult(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    papers: list[_BatchPaperExtraction]


_INSTRUCTIONS = """"""
_BATCH_INSTRUCTIONS = _INSTRUCTIONS + """

The input contains several papers. Return exactly one `papers` item for each
supplied paper identifier when possible. The paper_id must be copied exactly
from the input. Never use evidence from one paper in another paper's item.
If a paper cannot be extracted reliably, omit only that paper so it can be
retried individually; valid sibling items must still be returned.
"""


_GENERIC_DATASET_VALUES = {
    "dataset",
    "datasets",
    "benchmark dataset",
    "benchmark datasets",
    "widely used dataset",
    "widely used datasets",
    "widely used benchmark dataset",
    "widely used benchmark datasets",
    "public dataset",
    "public datasets",
}

_GENERIC_DATASET_KEYS = {
    canonical_evidence_key(value)
    for value in _GENERIC_DATASET_VALUES
}

_EXPLICIT_COMPARISON_PATTERN = re.compile(
    r"\b(?:compar(?:e|ed|ing|ison)|benchmark(?:ed|s)?|"
    r"evaluat(?:e|ed|ing)|test(?:ed|s|ing)?|outperform(?:ed|s|ing)?|"
    r"against|versus|vs\.?|baseline|relative to)\b",
    re.IGNORECASE,
)

_BACKGROUND_COMPARISON_PATTERN = re.compile(
    r"\b(?:background|related work|prior work|conventional|traditional|"
    r"existing approaches?|limitations?|slow|expensive|challenging|"
    r"shortcomings?)\b",
    re.IGNORECASE,
)

# These are implementation details only when a more informative focal method
# exists. Deliberately DO NOT include fine-tuning, pre-training, transfer
# learning, optimization, few-shot learning, etc. Those may themselves be the
# scientific intervention being studied.
_GENERIC_METHOD_DETAIL_PATTERN = re.compile(
    r"\b(?:implementation detail|preprocessing|pre-processing|"
    r"data augmentation|normalization|filtering|"
    r"learning[- ]rate scheduling|hyperparameter tuning|"
    r"postprocessing|post-processing)\b",
    re.IGNORECASE,
)

_SAMPLE_SIZE_PATTERN = re.compile(
    r"\b(?:dataset|subset|test set|training set|validation set|sample|cohort)"
    r"[^.!?]{0,40}?\b(?:of|with|containing|consisting of)?\s*"
    r"(\d[\d,]*)\s+"
    r"(images?|samples?|participants?|patients?|documents?|records?|instances?|examples?)\b",
    re.IGNORECASE,
)

_FUTURE_ACTION_PATTERN = re.compile(
    r"\b(?:assess|adapt|analy[sz]e|apply|benchmark|collect|compare|"
    r"conduct|develop|deploy|design|evaluate|examine|extend|explore|"
    r"improve|implement|investigate|measure|study|test|validate)\w*\b",
    re.IGNORECASE,
)




